In [1]:
!pip install -q sentence-transformers transformers accelerate gradio

In [2]:
from google.colab import files
uploaded = files.upload()
name = list(uploaded.keys())[0]
text = uploaded[name].decode("utf-8-sig")
print(name, len(text))

Saving test.txt to test.txt
test.txt 50267


In [3]:
import re, torch
from sentence_transformers import SentenceTransformer, util

def make_chunks(text, max_chars=600, overlap_sents=1):
    sents = [s.strip() for s in re.split(r'(?<=[.!?])\s+|\n+', text) if s.strip()]
    chunks, cur = [], []
    for s in sents:
        if cur and sum(len(x) for x in cur) + len(s) > max_chars:
            chunks.append(" ".join(cur))
            cur = cur[-overlap_sents:]
        cur.append(s)
    if cur:
        chunks.append(" ".join(cur))
    return chunks

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embedder.max_seq_length = 256

chunks = make_chunks(text)
vectors = embedder.encode(chunks, convert_to_tensor=True,
                          normalize_embeddings=True, show_progress_bar=True)
print(len(chunks), "чанков")

def retrieve(query, k=4):
    q = embedder.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    scores = util.cos_sim(q, vectors)[0]
    top = scores.topk(min(k, len(chunks)))
    return [(float(s), int(i)) for s, i in zip(top.values, top.indices)]

for s, i in retrieve("Что такое условная вероятность?", k=2):
    print(round(s, 2), chunks[i][:150], "\n")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

114 чанков
0.68 Оно описывает ее средний уровень в вероятностном смысле и играет важную роль в принятии решений в условиях неопределенности. Для дискретной случайной  

0.67 Если распределение имеет тяжелые хвосты или выраженную асимметрию, одной меры разброса может быть недостаточно. В таких случаях используют дополнитель 



In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
tok = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype, device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [13]:
SYSTEM = ("Ты помощник, который отвечает на вопросы строго по приведённому контексту. "
          "Не добавляй информацию, которой нет в контексте, не выдумывай определения и примеры. "
          "Если в контексте нет ответа или его части, прямо скажи об этом. "
          "Если вопрос состоит из нескольких частей, отвечай по пунктам. "
          "После каждого утверждения указывай номер фрагмента в квадратных скобках, например [2]. "
          "Отвечай по-русски.")

def answer(question, k=6):
    hits = retrieve(question, k)
    context = "\n\n".join(f"[{n}] {chunks[i]}" for n, (s, i) in enumerate(hits, 1))
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=700, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    ans = tok.decode(new_tokens, skip_special_tokens=True)
    if len(new_tokens) >= 700:
        ans += " […ответ обрезан по лимиту длины]"
    return ans, hits

NO_ANSWER = "В тексте этого нет."

def answer_safe(question, k=6):
    hits = retrieve(question, k)
    if hits[0][0] < THRESHOLD:
        return NO_ANSWER, hits
    return answer(question, k)

In [6]:
eval_set = [
    {"q": "Чем числовая характеристика результата случайного эксперимента отличается от обычной величины?", "key": "Случайные величины"},
    {"q": "Как называется средний уровень случайной величины в вероятностном смысле?", "key": "Математическое ожидание"},
    {"q": "Какими показателями измеряют разброс значений относительно среднего уровня?", "key": "Дисперсия и стандартное отклонение"},
    {"q": "Почему распределение суммы большого числа независимых величин часто оказывается близким к нормальному?", "key": "Центральная предельная теорема"},
    {"q": "Почему модель может хорошо работать на знакомых данных, но ошибаться на новых?", "key": "Переобучение и обобщающая способность"},
]

In [14]:
in_topic = [item["q"] for item in eval_set]
off_topic = [
    "Почему небо голубое?",
    "Кто выиграл чемпионат мира по футболу?",
    "Как приготовить борщ?",
    "Какая столица Франции?",
    "Как работает двигатель внутреннего сгорания?",
]

def best_score(q):
    return retrieve(q, k=1)[0][0]

on = [best_score(q) for q in in_topic]
off = [best_score(q) for q in off_topic]
print("по теме:    мин", round(min(on), 2), "макс", round(max(on), 2))
print("не по теме: мин", round(min(off), 2), "макс", round(max(off), 2))

THRESHOLD = round((min(on) + max(off)) / 2, 2)
print("порог:", THRESHOLD)
if min(on) <= max(off):
    print("Внимание: диапазоны пересекаются, порог будет ошибаться")

по теме:    мин 0.66 макс 0.77
не по теме: мин 0.13 макс 0.26
порог: 0.46


In [15]:
import numpy as np

def evaluate(eval_set, k=5):
    hits_n, rr = 0, []
    for item in eval_set:
        rank = None
        for r, (s, i) in enumerate(retrieve(item["q"], k), start=1):
            if item["key"].lower() in chunks[i].lower():
                rank = r
                break
        hits_n += rank is not None
        rr.append(1 / rank if rank else 0)
    return {"recall@5": hits_n / len(eval_set), "MRR": float(np.mean(rr))}

print(evaluate(eval_set))

{'recall@5': 1.0, 'MRR': 0.8}


In [17]:
import gradio as gr

def ui(q):
    ans, hits = answer_safe(q)
    src = "\n\n".join(f"[{n}] (сходство {s:.2f})\n{chunks[i]}"
                      for n, (s, i) in enumerate(hits, 1))
    return ans, src

demo = gr.Interface(
    fn=ui,
    inputs=gr.Textbox(label="Вопрос"),
    outputs=[gr.Textbox(label="Ответ"), gr.Textbox(label="Найденные фрагменты", lines=12)],
    title="RAG по тексту",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://068eeac319bb7bd082.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
